# Asymmetric Style Suppression Experiment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]

This notebook walks through the **asymmetric style suppression** experiment from [bergson](https://github.com/EleutherAI/bergson), a gradient-based data attribution library.

> **GPU requirements**: This experiment uses Qwen3-0.6B (no LoRA needed) and fits comfortably on **Colab Free** (T4, 15GB VRAM).

## Motivation

When using influence functions to trace model behavior back to training data, **style often dominates over semantics**. If your query is written in a different style than the training corpus (e.g., multi-choice eval vs. free-form training text), attribution will match on style rather than content.

This experiment evaluates strategies for suppressing style and recovering semantic matching, including:
- **Hessians** (train second moments, rank-1 style direction)
- **PCA projection** (projecting out dominant style directions)
- **Summed gradient** controls

## Setup

The experiment uses a synthetic dataset where facts appear in controlled writing styles (shakespeare vs. pirate), with train/eval splits designed so that style leakage and semantic accuracy are mutually exclusive.

In [ ]:
# Install bergson (includes all dependencies: torch, transformers, datasets, etc.)
!pip install -q bergson

## 1. Load the Dataset

The dataset is hosted on HuggingFace with pre-computed style rewrites. It has five splits:

| Split | Description |
|-------|-------------|
| `train` | 13,500 samples — 95% shakespeare (dominant), 5% pirate (minority) |
| `eval` | 4,500 samples — exclusive facts in pirate style (forces semantic matching) |
| `eval_majority_style` | Same facts as eval but in shakespeare style (control) |
| `eval_original_style` | Same facts in original unstyled text |
| `eval_pirate_style` | Same facts in pirate style |

Each sample has: `reworded` (stylized text), `fact` (original), `style`, `identifier` (which person), `field` (which attribute), and `template` (which phrasing).

In [ ]:
from pathlib import Path

from datasets import load_dataset

HF_DATASET = "EleutherAI/bergson-asymmetric-style"
MODEL = "Qwen/Qwen3-0.6B-Base"
BASE_PATH = Path("runs/style_ablation")
BASE_PATH.mkdir(parents=True, exist_ok=True)

# Load all splits
all_splits = load_dataset(HF_DATASET)
train_ds = all_splits["train"]
eval_ds = all_splits["eval"]
eval_majority_ds = all_splits["eval_majority_style"]

# Save to disk for bergson CLI
train_path = BASE_PATH / "train.hf"
eval_path = BASE_PATH / "eval.hf"
eval_majority_path = BASE_PATH / "eval_majority_style.hf"

if not train_path.exists():
    train_ds.save_to_disk(str(train_path))
if not eval_path.exists():
    eval_ds.save_to_disk(str(eval_path))
if not eval_majority_path.exists():
    eval_majority_ds.save_to_disk(str(eval_majority_path))

print(f"Train: {len(train_ds)} samples")
print(f"  Styles: {dict(zip(*map(list, zip(*train_ds.to_pandas()['style'].value_counts().items()))))}")
print(f"Eval:  {len(eval_ds)} samples (minority style queries)")
print("\nSample train row:")
print(train_ds[0])

### Understand the dataset design

The key insight is **disjoint partitioning**: some facts only appear in the dominant (shakespeare) style in training. When we query with a pirate-style version of that fact, the *only* way to find the right training sample is semantic matching — style matching would point to the wrong samples.

- **Exclusive facts** (~50%): Only in dominant style in training. Eval queries use the minority style for these.
- **Shared facts** (~50%): In both styles. These are the "decoys" that style-matching would find.

In [ ]:
# Look at how the style split works

train_df = train_ds.to_pandas()
eval_df = eval_ds.to_pandas()

# Count style distribution in training
print("Training style distribution:")
print(train_df["style"].value_counts())
print(f"\nDominant ratio: {(train_df['style'] == 'shakespeare').mean():.1%}")

# The eval set queries exclusive facts in minority style
# The ground truth match is always in the dominant style
print(f"\nEval: all queries are in '{eval_df['style'].iloc[0]}' style")
print(f"Expected match style: '{eval_df['expected_match_style'].iloc[0]}' (dominant)")

# Show a concrete example
idx = 0
eval_fact = eval_df.iloc[idx]
matching_train = train_df[
    (train_df["identifier"] == eval_fact["identifier"]) &
    (train_df["field"] == eval_fact["field"])
]
print("\n--- Example ---")
print(f"Eval query (pirate): {eval_fact['reworded'][:120]}...")
print(f"Correct match (shakespeare): {matching_train.iloc[0]['reworded'][:120]}...")
print(f"Same fact? identifier={eval_fact['identifier']}, field={eval_fact['field']}")

## 2. Build the Gradient Index (Training Set)

This is the most expensive step: we collect per-sample gradients for every training example. bergson uses hook-based gradient collection — for each linear layer, it captures `P = g^T @ a` (the outer product of the output gradient and the input activation), optionally projected to lower dimension.

We use `bergson build` via the CLI, which handles:
1. Loading the model and tokenizer
2. Discovering all linear layers
3. Token-budget batching (packing samples to maximize GPU utilization)
4. Per-sample gradient collection with random projection
5. Computing second-moment hessians

In [ ]:
import subprocess

INDEX_PATH = str(BASE_PATH / "index")
PROJECTION_DIM = 16  # Random projection dimension (keeps gradients small)
TOKEN_BATCH_SIZE = 2048  # Token budget per batch

if not Path(INDEX_PATH).exists():
    cmd = [
        "bergson", "build", INDEX_PATH,
        "--model", MODEL,
        "--dataset", str(train_path),
        "--prompt_column", "reworded",
        "--drop_columns",
        "--projection_dim", str(PROJECTION_DIM),
        "--token_batch_size", str(TOKEN_BATCH_SIZE),
        "--skip_hessians", "False",
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError("bergson build failed")
    print("Done! Index built at", INDEX_PATH)
else:
    print(f"Index already exists at {INDEX_PATH}, skipping...")

### Inspect the index

Let's look at what `bergson build` produced. The gradients are stored as a memory-mapped structured numpy array — each field corresponds to a linear layer in the model.

In [ ]:
import json

import numpy as np

from bergson.data import load_gradients

# Load index metadata
with open(Path(INDEX_PATH) / "info.json") as f:
    info = json.load(f)

print(f"Number of gradient vectors: {info['num_grads']}")
print(f"Module names ({len(info['grad_sizes'])} modules):")
for name, size in list(info["grad_sizes"].items())[:5]:
    print(f"  {name}: {size} dims")
print(f"  ... ({len(info['grad_sizes'])} total)")
print(f"\nTotal gradient dimension: {sum(info['grad_sizes'].values())}")

## 3. Collect Eval Gradients

Now we collect gradients for the eval queries using the **same projection** as the training index. This ensures the gradient spaces are aligned.

In [ ]:
def run_bergson_build(output_path, dataset_path, prompt_column="reworded", completion_column=""):
    """Helper to run bergson build for eval gradient collection."""
    if Path(output_path).exists():
        print(f"Already exists: {output_path}, skipping...")
        return

    # Read the index config to match projection settings
    with open(Path(INDEX_PATH) / "index_config.json") as f:
        idx_cfg = json.load(f)

    cmd = [
        "bergson", "build", str(output_path),
        "--model", MODEL,
        "--dataset", str(dataset_path),
        "--prompt_column", prompt_column,
        "--drop_columns",
        "--projection_dim", str(idx_cfg["projection_dim"]),
        "--token_batch_size", str(TOKEN_BATCH_SIZE),
        "--skip_hessians", "True",
    ]
    if completion_column:
        cmd.extend(["--completion_column", completion_column])

    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError(f"bergson build failed for {output_path}")
    print(f"Done: {output_path}")

# Collect eval gradients (minority style - pirate)
EVAL_GRADS_PATH = str(BASE_PATH / "eval_grads")
run_bergson_build(EVAL_GRADS_PATH, eval_path)

# Collect majority style eval gradients (shakespeare - control)
EVAL_MAJORITY_GRADS_PATH = str(BASE_PATH / "eval_grads_majority")
run_bergson_build(EVAL_MAJORITY_GRADS_PATH, eval_majority_path)

## 4. Load Gradients and Score

Now we do the core attribution step manually. For each eval query, we compute cosine similarity against all training gradients:

```
score(query, train_i) = normalize(g_query) · normalize(g_train_i)
```

The gradients are stored per-module in bfloat16. We'll convert to float32 and concatenate across modules to get a single vector per sample.

In [ ]:
import ml_dtypes
import torch


def load_grads_as_float(grads_path, module_names):
    """Load structured gradients and concatenate across modules as float32."""
    grads = load_gradients(grads_path, structured=True)
    parts = []
    for name in module_names:
        g = grads[name]
        # Stored as bfloat16 (2-byte void) — convert to float32
        if g.dtype == np.dtype("|V2"):
            g = g.view(ml_dtypes.bfloat16).astype(np.float32)
        parts.append(g)
    return np.concatenate(parts, axis=1)

# Get module names from index metadata
module_names = list(info["grad_sizes"].keys())
print(f"Loading gradients for {len(module_names)} modules...")

# Load and concatenate train gradients
train_grad_matrix = load_grads_as_float(INDEX_PATH, module_names)
print(f"Train gradients: {train_grad_matrix.shape}")  # (n_train, total_dim)

# Load eval gradients (minority style)
eval_grad_matrix = load_grads_as_float(EVAL_GRADS_PATH, module_names)
print(f"Eval gradients:  {eval_grad_matrix.shape}")  # (n_eval, total_dim)

# Load majority style eval gradients (for summed_eval later)
eval_majority_grad_matrix = load_grads_as_float(EVAL_MAJORITY_GRADS_PATH, module_names)
print(f"Eval majority:   {eval_majority_grad_matrix.shape}")

In [ ]:
def cosine_similarity_scores(eval_grads, train_grads):
    """Compute cosine similarity between eval and train gradient matrices.

    Args:
        eval_grads: (n_eval, dim) numpy array
        train_grads: (n_train, dim) numpy array

    Returns:
        (n_eval, n_train) numpy array of cosine similarities
    """
    # Convert to torch for efficient GPU computation
    eval_t = torch.from_numpy(eval_grads).cuda()
    train_t = torch.from_numpy(train_grads).cuda()

    # Unit normalize
    eval_t = eval_t / (eval_t.norm(dim=1, keepdim=True) + 1e-8)
    train_t = train_t / (train_t.norm(dim=1, keepdim=True) + 1e-8)

    # Cosine similarity via matrix multiply
    scores = (eval_t @ train_t.T).cpu().numpy()
    return scores

## 5. Compute Metrics

For each eval query, we check the top-k training matches:
- **Semantic accuracy**: Does the top match share the same `(identifier, field)`? This means attribution found the right fact.
- **Style leakage**: Is the top match from the minority style? If so, attribution matched on style, not content.

In [ ]:
MINORITY_STYLE = "pirate"

def compute_metrics(scores, train_ds, eval_ds, top_k=5):
    """Compute semantic accuracy and style leakage from a score matrix.

    Args:
        scores: (n_eval, n_train) similarity scores
        train_ds: training dataset with identifier, field, style columns
        eval_ds: eval dataset with identifier, field columns

    Returns:
        dict with metric values
    """
    train_ids = list(zip(train_ds["identifier"], train_ds["field"]))
    train_styles = list(train_ds["style"])
    eval_ids = list(zip(eval_ds["identifier"], eval_ds["field"]))

    top1_semantic = 0
    top5_semantic = 0
    top1_leak = 0

    n_eval = scores.shape[0]
    for i in range(n_eval):
        top_indices = np.argsort(scores[i])[::-1][:top_k]

        # Top-1 semantic accuracy
        if train_ids[top_indices[0]] == eval_ids[i]:
            top1_semantic += 1

        # Top-5 semantic recall
        if any(train_ids[top_indices[j]] == eval_ids[i] for j in range(top_k)):
            top5_semantic += 1

        # Top-1 style leakage
        if train_styles[top_indices[0]] == MINORITY_STYLE:
            top1_leak += 1

    return {
        "top1_semantic": top1_semantic / n_eval,
        "top5_semantic_recall": top5_semantic / n_eval,
        "top1_leak": top1_leak / n_eval,
    }

# Compute baseline metrics
results = {}
scores_no_hess = cosine_similarity_scores(eval_grad_matrix, train_grad_matrix)
results["no_hess"] = compute_metrics(scores_no_hess, train_ds, eval_ds)
del scores_no_hess

m = results["no_hess"]
print("Baseline (no hessian):")
print(f"  Top-1 Semantic Accuracy: {m['top1_semantic']:.2%}")
print(f"  Top-5 Semantic Recall:   {m['top5_semantic_recall']:.2%}")
print(f"  Top-1 Style Leakage:     {m['top1_leak']:.2%}")
print("\n→ High leakage = attribution is matching on style, not semantics")

## 6. Majority Style Control

As a sanity check, we score using majority-style (shakespeare) eval queries. Since these match the training distribution, there's no style mismatch — attribution should work well.

In [ ]:
print("Computing majority style control scores...")
scores_majority = cosine_similarity_scores(eval_majority_grad_matrix, train_grad_matrix)
results["majority_no_hess"] = compute_metrics(scores_majority, train_ds, eval_majority_ds)
del scores_majority, eval_majority_grad_matrix

m = results["majority_no_hess"]
print("Majority style control (no style mismatch):")
print(f"  Top-1 Semantic Accuracy: {m['top1_semantic']:.2%}")
print(f"  Top-5 Semantic Recall:   {m['top5_semantic_recall']:.2%}")
print(f"  Top-1 Style Leakage:     {m['top1_leak']:.2%}")
print("\n→ This is the upper bound — when styles match, attribution works")

## 7. Hessian Strategies

Preconditioning transforms gradients before scoring: `score(q, t) = (g_q @ H^{-1}) · g_t`

This can downweight common/systematic directions.

### Train second moment (standard influence functions)
`H = G_train^T @ G_train` — the standard influence function hessian (an approximation to the Fisher information matrix). This downweights directions that are common across training data, which often correspond to style rather than content.

In [ ]:
from bergson.gradients import GradientProcessor
from bergson.utils.math import psd_rsqrt


def apply_hessian_per_module(eval_grads_path, hess, module_names, grad_sizes, damping=0.1):
    """Apply per-module hessian inverse to eval gradients.

    For each module, computes: g_preconditioned = g @ H^{-1/2}
    where H is the hessian for that module, regularized with damping.

    Args:
        eval_grads_path: Path to eval gradients
        hess: dict mapping module name -> hessian tensor H
        module_names: list of module names
        grad_sizes: dict mapping module name -> gradient dimension
        damping: regularization strength

    Returns:
        (n_eval, total_dim) numpy array of preconditioned gradients
    """
    grads = load_gradients(eval_grads_path, structured=True)
    parts = []

    for name in module_names:
        g = grads[name]
        if g.dtype == np.dtype("|V2"):
            g = g.view(ml_dtypes.bfloat16).astype(np.float32)

        if name in hess:
            H = hess[name].float().cuda()
            # Add damping: H_damped = H + damping * I
            H_damped = H + damping * torch.eye(H.shape[0], device=H.device)
            # Compute H^{-1/2}
            H_inv_sqrt = psd_rsqrt(H_damped)
            # Apply: g_new = g @ H^{-1/2}
            g_t = torch.from_numpy(g).float().cuda()
            g_t = g_t @ H_inv_sqrt
            g = g_t.cpu().numpy()

        parts.append(g)

    return np.concatenate(parts, axis=1)

# Load the train second moment hessian (computed during bergson build)
processor = GradientProcessor.load(INDEX_PATH, map_location="cpu")
train_hess = processor.hessians
print(f"Loaded train hessian with {len(train_hess)} modules")
print(f"Example shape: {list(train_hess.values())[0].shape}")

In [ ]:
# PCA ablation helper
def pca_ablate_eval(eval_grads, k=100):
    """Project eval gradients orthogonal to their top-k PCA components."""
    g = torch.from_numpy(eval_grads).float().cuda()
    g_centered = g - g.mean(dim=0, keepdim=True)
    U, S, Vh = torch.linalg.svd(g_centered, full_matrices=False)
    V_k = Vh[:k].T  # (dim, k)
    proj = g @ V_k @ V_k.T
    result = (g - proj).cpu().numpy()
    del g, g_centered, U, S, Vh, V_k, proj
    torch.cuda.empty_cache()
    return result

# Strategy: Train second moment hessian (standard influence functions)
print("Applying train second moment hessian to eval gradients...")
eval_hess_index = apply_hessian_per_module(
    EVAL_GRADS_PATH, train_hess, module_names, info["grad_sizes"]
)
scores_index = cosine_similarity_scores(eval_hess_index, train_grad_matrix)
results["influence_fn"] = compute_metrics(scores_index, train_ds, eval_ds)
del scores_index, eval_hess_index

m = results["influence_fn"]
print("Train second moment (standard influence functions):")
print(f"  Top-1 Semantic Accuracy: {m['top1_semantic']:.2%}")
print(f"  Top-1 Style Leakage:     {m['top1_leak']:.2%}")

# Free hessian
del train_hess, processor
torch.cuda.empty_cache()

## 8. PCA Ablation

An alternative to preconditioning: project out the top-k principal components of the eval gradient matrix. These top components tend to capture systematic directions (often style), so removing them preserves semantic signal while suppressing style leakage.

In [ ]:
# Strategy: PCA ablation on raw gradients
print("Computing PCA ablation (k=100) on raw eval gradients...")
eval_pca_raw = pca_ablate_eval(eval_grad_matrix, k=100)
scores_pca_raw = cosine_similarity_scores(eval_pca_raw, train_grad_matrix)
results["pca_k100"] = compute_metrics(scores_pca_raw, train_ds, eval_ds)
del eval_pca_raw, scores_pca_raw

m = results["pca_k100"]
print("PCA ablation (k=100):")
print(f"  Top-1 Semantic Accuracy: {m['top1_semantic']:.2%}")
print(f"  Top-1 Style Leakage:     {m['top1_leak']:.2%}")

### Influence Functions + PCA Ablation

Combining the train second moment hessian with PCA ablation gives the best results — but requires holding both the preconditioned and raw gradient matrices in RAM simultaneously. **This cell requires more RAM than Colab Free provides** (~25 GB); skip it if running on a free T4.

In [ ]:
# Strategy: Influence functions + PCA ablation
# Requires reloading the hessian since we freed it above
processor = GradientProcessor.load(INDEX_PATH, map_location="cpu")
train_hess = processor.hessians

print("Applying influence function hessian + PCA ablation (k=100)...")
eval_hess_index = apply_hessian_per_module(
    EVAL_GRADS_PATH, train_hess, module_names, info["grad_sizes"]
)
del train_hess, processor

eval_if_pca = pca_ablate_eval(eval_hess_index, k=100)
del eval_hess_index
scores_if_pca = cosine_similarity_scores(eval_if_pca, train_grad_matrix)
results["influence_fn_pca_k100"] = compute_metrics(scores_if_pca, train_ds, eval_ds)
del eval_if_pca, scores_if_pca
torch.cuda.empty_cache()

m = results["influence_fn_pca_k100"]
print("Influence functions + PCA ablation (k=100):")
print(f"  Top-1 Semantic Accuracy: {m['top1_semantic']:.2%}")
print(f"  Top-1 Style Leakage:     {m['top1_leak']:.2%}")

## 9. Results Summary

### Results Table

Each strategy is evaluated on three key metrics:
- **Top-1 Semantic Accuracy**: Does the top match share the same underlying fact? (higher is better)
- **Top-5 Semantic Recall**: Do any of the top-5 matches share the same fact? (higher is better)
- **Top-1 Style Leakage**: Is the top match from the minority style? (lower is better)

Due to the disjoint partitioning, high style leakage implies low semantic accuracy and vice versa.

In [ ]:
sorted_results = sorted(results.items(), key=lambda x: -x[1]["top1_semantic"])

print(f"{'Strategy':<25} {'Top-1 Sem':<12} {'Top-5 Recall':<13} {'Top-1 Leak':<12}")
print("-" * 62)
for name, m in sorted_results:
    print(
        f"{name:<25} {m['top1_semantic']:<12.2%} "
        f"{m['top5_semantic_recall']:<13.2%} {m['top1_leak']:<12.2%}"
    )

### Visualize Results

In [ ]:
import matplotlib.pyplot as plt

names = [name for name, _ in sorted_results]
top1_sem = [m["top1_semantic"] * 100 for _, m in sorted_results]
top1_leak = [m["top1_leak"] * 100 for _, m in sorted_results]

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(names))
bar_width = 0.35

bars1 = ax.bar([i - bar_width / 2 for i in x], top1_sem, bar_width, label="Top-1 Semantic Acc (%)")
bars2 = ax.bar([i + bar_width / 2 for i in x], top1_leak, bar_width, label="Top-1 Style Leak (%)")

ax.set_ylabel("Percentage")
ax.set_title("Style Suppression Strategies: Semantic Accuracy vs Style Leakage")
ax.set_xticks(list(x))
ax.set_xticklabels(names, rotation=45, ha="right", fontsize=9)
ax.legend()
ax.set_ylim(0, 105)

fig.tight_layout()
plt.show()

## 10. Key Takeaways

The fundamental insight is that **the problem isn't about finding the right hessian** — it's about the **query gradient itself being a style outlier**.

When the eval data is in a minority style (5% of training), its gradients are dominated by style-specific components that match the small minority-style portion of training rather than the semantically-correct majority-style matches.

### Strategy descriptions

| Strategy | Description |
|----------|-------------|
| `no_hess` | Bare gradient cosine similarity. Style dominates. |
| `majority_no_hess` | Query in the majority style — no style mismatch. Upper bound. |
| `influence_fn` | Standard influence function hessian (`G_train^T @ G_train`). |
| `pca_k100` | Project out top-100 PCA components from eval gradients. |
| `influence_fn_pca_k100` | Standard influence functions + top-100 PCA ablation. |

## 11. Try It Yourself

The cells above expose all the building blocks. Here are some things to experiment with:

- **Change the PCA ablation rank**: Try different `k` values in `pca_ablate_eval` — smaller k removes less style signal, larger k risks removing semantic signal too
- **Change the projection dimension**: Try `PROJECTION_DIM = 0` (full gradients) or `32` and re-run
- **Try different damping values**: The `damping` parameter in `apply_hessian_per_module` controls regularization strength
- **Build a custom hessian**: Use the gradient loading code to compute any matrix `H` and pass it to `apply_hessian_per_module`
- **Change the model**: Try a larger model like `Qwen/Qwen3-1.7B-Base` (needs ~8GB VRAM) to see if the effect scales

In [ ]:
# Final summary with all strategies
sorted_results = sorted(results.items(), key=lambda x: -x[1]["top1_semantic"])

print(f"\n{'='*62}")
print("FINAL RESULTS")
print(f"{'='*62}")
print(f"{'Strategy':<25} {'Top-1 Sem':<12} {'Top-5 Recall':<13} {'Top-1 Leak':<12}")
print("-" * 62)
for name, m in sorted_results:
    print(
        f"{name:<25} {m['top1_semantic']:<12.2%} "
        f"{m['top5_semantic_recall']:<13.2%} {m['top1_leak']:<12.2%}"
    )

# Save results
results_path = BASE_PATH / "experiment_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {results_path}")